In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
 
from scipy.linalg import eigh
import scipy.linalg as la
import scipy.sparse as sps
from scipy.spatial import distance_matrix
import scipy.spatial as spatial

from sklearn.preprocessing import StandardScaler

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

Synthetic data Experiment 1 - Spatially separated clusters

In [ ]:
## HELPER FUNCTIONS

# Define clustered graph structure
def generate_clustered_graph(n_clusters=3, nodes_per_cluster=[30, 40, 30], cluster_separation=5.0, random_state=None):
    rng = np.random.default_rng(random_state)
    positions = []
    cluster_labels = []
    centers = np.linspace(0, (n_clusters - 1) * cluster_separation, n_clusters)
    for i, n_nodes in enumerate(nodes_per_cluster):
        cluster_center = np.array([centers[i], 0])
        cluster_points = cluster_center + rng.normal(scale=1.0, size=(n_nodes, 2))
        positions.append(cluster_points)
        cluster_labels.extend([i] * n_nodes)
    positions = np.vstack(positions)
    cluster_labels = np.array(cluster_labels)
    # Build adjacency: connect nodes within radius r (spatial proximity)
    D = distance_matrix(positions, positions)
    r = 2.0  # radius threshold
    A = (D < r).astype(float)
    np.fill_diagonal(A, 0)
    return A, positions, cluster_labels

# Compute covariance matrix from adjacency
def compute_covariance(A, params={'t': 1.0, 'epsilon': 1e-5}):
    D = np.diag(A.sum(axis=1))
    L = D - A  # graph Laplacian
    n = A.shape[0]
    # Heat kernel covariance: exp(-t * L)
    eigvals, eigvecs = eigh(L)
    C = eigvecs @ np.diag(np.exp(-params.get('t', 1.0) * eigvals)) @ eigvecs.T
    return C

# Generate Gaussian Random Field samples (X, y)
def generate_graph_grf_dataset(A, cluster_labels, n_samples=1000, cov_params={'t': 1.0}, signal_strength=1.0, pos_fraction=0.5, random_state=None):
    rng = np.random.default_rng(random_state)
    n_nodes = A.shape[0]
    # Compute covariance matrix
    Sigma = compute_covariance(A, params=cov_params)
    # Define class-dependent means
    unique_clusters = np.unique(cluster_labels)
    cluster_signal = rng.normal(scale=signal_strength, size=len(unique_clusters))
    mu0 = np.zeros(n_nodes)
    mu1 = np.array([cluster_signal[c] for c in cluster_labels])
    # Generate binary labels
    y = rng.binomial(1, pos_fraction, size=n_samples)
    # Generate samples from GRF
    X = np.array([rng.multivariate_normal(mu1 if yi == 1 else mu0, Sigma) for yi in y])
    return X, y, mu0, mu1, Sigma

# Plot
def visualize_graph(positions, A, cluster_labels):
    G = nx.from_numpy_array(A)
    plt.figure(figsize=(6, 4))
    nx.draw(G, pos={i: positions[i] for i in range(len(positions))},
            node_color=cluster_labels, cmap='tab10', with_labels=False, node_size=60)
    plt.title('Clustered Graph Structure')
    plt.show()
def visualize_mean_field(positions, mu1):
    plt.figure(figsize=(6, 4))
    plt.scatter(positions[:, 0], positions[:, 1], c=mu1, cmap='coolwarm', s=60)
    plt.colorbar(label='Mean value (μ₁)')
    plt.title('Class 1 Mean Field over Graph Nodes')
    plt.show()
def visualize_samples(X, y, n_show=5):
    plt.figure(figsize=(10, 4))
    colors = ['red' if label == 1 else 'blue' for label in y[:n_show]]
    for i in range(n_show):
        plt.plot(X[i], color=colors[i], alpha=0.7)
    plt.xlabel('Node Index')
    plt.ylabel('Feature Value')
    plt.legend()
    plt.title('Example GRF Samples')
    plt.show()

In [ ]:
## RUN TO GENERATE DATA 

if __name__ == "__main__":
    # Generate graph 
    A, positions, clusters = generate_clustered_graph(n_clusters=4,nodes_per_cluster=[250, 250, 250, 250],cluster_separation=5,random_state=42)
    # Generate tabular dataset
    X, y, mu0, mu1, Sigma = generate_graph_grf_dataset(A, clusters,n_samples=1000,cov_params={'t': 1},signal_strength=1,pos_fraction=0.5,random_state=42)
    # Plot to better understand data composition
    visualize_graph(positions, A, clusters)
    visualize_mean_field(positions, mu1)
    visualize_samples(X, y, n_show = 50) # adjust no. of samples to show, the trends shold be the same
    print(f"Generated dataset shape: X={X.shape}, y={y.shape}")

Synthetic data Experiment 2 - Clusters separated by edge probability

In [ ]:
## HELPER FUNCTIONS

# Generate graph 
def generate_sbm_graph(block_sizes=[30, 40, 30],
                       p_in=0.2, # within-cluster edges
                       p_between=0.01, # between-cluster edges
                       random_state=None,
                       return_graph=False):
    rng = np.random.default_rng(random_state)
    block_sizes = list(block_sizes)
    n_blocks = len(block_sizes)
    n = sum(block_sizes)
    p_matrix = np.full((n_blocks, n_blocks), p_between, dtype=float)
    for i in range(n_blocks):
        p_matrix[i, i] = p_in
    G = nx.stochastic_block_model(sizes=block_sizes, p=p_matrix, seed=int(rng.integers(1<<30)))
    A = nx.to_numpy_array(G, dtype=float)
    np.fill_diagonal(A, 0.0)
    cluster_labels = np.concatenate([[i]*s for i, s in enumerate(block_sizes)])
    pos_dict = nx.spring_layout(G, seed=int(rng.integers(1<<30)))
    positions = np.vstack([pos_dict[i] for i in range(n)])
    if return_graph:
        return A, positions, cluster_labels, G
    return A, positions, cluster_labels

# Tabular dataset generation is the same as above

# Compute covariance matrix from adjacency
def compute_covariance(A, params={'t': 1.0, 'epsilon': 1e-5}):
    D = np.diag(A.sum(axis=1))
    L = D - A  # graph Laplacian
    n = A.shape[0]
    eigvals, eigvecs = eigh(L)
    C = eigvecs @ np.diag(np.exp(-params.get('t', 1.0) * eigvals)) @ eigvecs.T
    return C

#  Generate Gaussian Random Field samples (X, y)
def generate_graph_grf_dataset(A, cluster_labels, n_samples=1000, cov_method='heat', cov_params={'t': 1.0}, signal_strength=1.0, pos_fraction=0.5, random_state=None):
    rng = np.random.default_rng(random_state)
    n_nodes = A.shape[0]
    Sigma = compute_covariance(A, method=cov_method, params=cov_params)
    unique_clusters = np.unique(cluster_labels)
    cluster_signal = rng.normal(scale=signal_strength, size=len(unique_clusters))
    mu0 = np.zeros(n_nodes)
    mu1 = np.array([cluster_signal[c] for c in cluster_labels])
    y = rng.binomial(1, pos_fraction, size=n_samples)
    X = np.array([rng.multivariate_normal(mu1 if yi == 1 else mu0, Sigma) for yi in y])
    return X, y, mu0, mu1, Sigma

# Plot

def visualize_graph(positions, A, cluster_labels):
    plt.figure(figsize=(8, 5))
    plt.scatter(positions[:, 0], positions[:, 1], c=cluster_labels, cmap='tab10', s=30, zorder=2)
    n = A.shape[0]
    edges = np.argwhere(A > 0.5)  # since A is 0/1 adjacency
    for i, j in edges:
        if i < j:  # draw each undirected edge once
            plt.plot([positions[i, 0], positions[j, 0]],[positions[i, 1], positions[j, 1]],color='gray',alpha=0.2, linewidth=0.5, zorder=1)
    plt.title("SBM nodes (colored by block) — all edges shown", fontsize=12)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
def visualize_mean_field(positions, mu1):
    plt.figure(figsize=(6, 4))
    plt.scatter(positions[:, 0], positions[:, 1], c=mu1, cmap='coolwarm', s=60)
    plt.colorbar(label='Mean value (μ₁)')
    plt.title('Class 1 Mean Field over Graph Nodes')
    plt.show()
def visualize_samples(X, y, n_show=5):
    plt.figure(figsize=(10, 4))
    colors = ['red' if label == 1 else 'blue' for label in y[:n_show]]
    for i in range(n_show):
        plt.plot(X[i], color=colors[i], alpha=0.7, label=f'Sample {i}, y={y[i]}')
    plt.xlabel('Node Index')
    plt.ylabel('Feature Value')
    plt.legend()
    plt.title('Example GRF Samples')
    plt.show()

In [ ]:
## RUN TO GENERATE DATA 
if __name__ == "__main__":
    A, positions, cluster_labels = generate_sbm_graph(block_sizes=[50, 50, 50, 50], p_in=0.01, p_between=0.01, random_state=42)
    X, y, mu0, mu1, Sigma = generate_graph_grf_dataset(A, cluster_labels, n_samples=1000, cov_params={'t': 1}, signal_strength=1, pos_fraction=0.5, random_state=42)
    visualize_graph(positions, A, cluster_labels)
    visualize_mean_field(positions, mu1)
    visualize_samples(X, y, n_show = 50)

The following objects represent the synthetic data generated in this notebook

X -> features

y -> labels 

A -> adjacency matrix

positions -> node positions

clusters -> cluster labels for each node